# 🌫️ Exploración de Datos — Calidad del Aire

**Curso:** Big Data – Primer Parcial  
**Tema:** Monitoreo de Calidad del Aire con Técnicas de Big Data  
**Fuente de datos:** OpenAQ (https://openaq.org)  

---

## 📋 Contenido del Notebook

1. Importación de librerías
2. Carga y vista previa del dataset
3. Análisis exploratorio (EDA)
4. Limpieza y transformación de datos
5. Visualizaciones
6. Conclusiones y por qué se necesita Big Data

---
## 1️⃣ Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os
import random
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Estilo visual
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('✅ Librerías cargadas correctamente')
print(f'   Pandas  : {pd.__version__}')
print(f'   NumPy   : {np.__version__}')

---
## 2️⃣ Carga del Dataset

Primero intentamos cargar datos reales generados por el script de ingesta.  
Si no existen, generamos un dataset simulado para continuar el análisis.

In [ ]:
def generar_dataset_simulado(n=5000):
    """
    Genera un dataset simulado realista de calidad del aire.
    Simula lecturas de 10 sensores durante varios meses.
    """
    random.seed(42)
    np.random.seed(42)
    
    sensores    = [f'Sensor-{i:02d}' for i in range(1, 11)]
    parametros  = ['pm25', 'pm10', 'no2', 'o3', 'co']
    unidades    = {'pm25': 'µg/m³', 'pm10': 'µg/m³', 'no2': 'ppb', 'o3': 'ppb', 'co': 'ppm'}
    base_valor  = {'pm25': 35, 'pm10': 60, 'no2': 25, 'o3': 45, 'co': 0.8}
    
    fecha_inicio = datetime(2025, 1, 1)
    registros = []
    
    for i in range(n):
        parametro = random.choice(parametros)
        sensor    = random.choice(sensores)
        fecha     = fecha_inicio + timedelta(minutes=5 * i)
        
        # Simular patrón horario (peor en horas pico)
        hora = fecha.hour
        factor_hora = 1.5 if (7 <= hora <= 9 or 17 <= hora <= 20) else 1.0
        valor = round(base_valor[parametro] * factor_hora * np.random.lognormal(0, 0.3), 2)
        
        registros.append({
            'fecha_utc':  fecha,
            'valor':      valor,
            'parametro':  parametro,
            'unidad':     unidades[parametro],
            'ubicacion':  sensor,
            'ciudad':     'Tegucigalpa',
            'pais':       'HN',
        })
    
    return pd.DataFrame(registros)


# Intentar cargar datos reales
ruta_raw = '../data/raw/calidad_aire_tegucigalpa.csv'

if os.path.exists(ruta_raw):
    df = pd.read_csv(ruta_raw, parse_dates=['fecha_utc'])
    print(f'✅ Datos reales cargados desde: {ruta_raw}')
else:
    df = generar_dataset_simulado(5000)
    print('ℹ️  Dataset simulado generado (no se encontró archivo real)')
    print('   Para datos reales, ejecuta primero: python scripts/ingesta_incremental.py')

print(f'\n📊 Shape del dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas')

---
## 3️⃣ Análisis Exploratorio (EDA)

In [ ]:
# Vista previa
print('🔍 Primeras 5 filas:')
df.head()

In [ ]:
# Información general
print('📋 Información del Dataset:')
df.info()

In [ ]:
# Estadísticas descriptivas
print('📈 Estadísticas descriptivas:')
df.describe()

In [ ]:
# Conteo por parámetro
print('🔢 Registros por parámetro:')
print(df['parametro'].value_counts().to_string())

print('\n🏙️ Conteo por ciudad:')
print(df['ciudad'].value_counts().to_string())

print('\n🔌 Conteo por sensor:')
print(df['ubicacion'].value_counts().to_string())

In [ ]:
# Valores nulos
print('❓ Valores nulos por columna:')
nulos = df.isnull().sum()
print(nulos[nulos >= 0].to_string())
print(f'\nTotal nulos: {df.isnull().sum().sum()}')

---
## 4️⃣ Limpieza y Transformación de Datos

In [ ]:
# 1. Eliminar duplicados
antes = len(df)
df = df.drop_duplicates(subset=['fecha_utc', 'ubicacion', 'parametro'])
print(f'🧹 Duplicados eliminados: {antes - len(df)}')

# 2. Eliminar valores negativos (imposibles físicamente)
df = df[df['valor'] >= 0]

# 3. Eliminar outliers extremos (valores > percentil 99.5)
p995 = df['valor'].quantile(0.995)
df_limpio = df[df['valor'] <= p995].copy()
print(f'🔍 Outliers eliminados: {len(df) - len(df_limpio)}')
df = df_limpio

# 4. Extraer componentes de fecha
df['hora']      = df['fecha_utc'].dt.hour
df['dia_semana']= df['fecha_utc'].dt.day_name()
df['mes']       = df['fecha_utc'].dt.month
df['fecha']     = df['fecha_utc'].dt.date

print(f'\n✅ Dataset limpio: {df.shape[0]:,} registros')
df.head(3)

In [ ]:
# Guardar datos procesados
ruta_procesado = '../data/processed/calidad_aire_limpio.csv'
os.makedirs('../data/processed', exist_ok=True)
df.to_csv(ruta_procesado, index=False)
print(f'💾 Datos limpios guardados en: {ruta_procesado}')

---
## 5️⃣ Visualizaciones

In [ ]:
# ── Gráfico 1: Distribución de valores por parámetro ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
df.boxplot(column='valor', by='parametro', ax=axes[0], grid=True)
axes[0].set_title('Distribución de Valores por Parámetro')
axes[0].set_xlabel('Contaminante')
axes[0].set_ylabel('Concentración')
plt.sca(axes[0])
plt.xticks(rotation=45)

# Conteo de registros
conteo = df['parametro'].value_counts()
axes[1].bar(conteo.index, conteo.values, color=sns.color_palette('husl', len(conteo)))
axes[1].set_title('Cantidad de Registros por Parámetro')
axes[1].set_xlabel('Contaminante')
axes[1].set_ylabel('Número de Registros')
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('../docs/grafico_distribucion.png', dpi=120, bbox_inches='tight')
plt.show()
print('💾 Gráfico guardado en docs/')

In [ ]:
# ── Gráfico 2: Patrón horario del PM2.5 ──
df_pm25 = df[df['parametro'] == 'pm25']

if len(df_pm25) > 0:
    promedio_hora = df_pm25.groupby('hora')['valor'].mean()

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(promedio_hora.index, promedio_hora.values,
            marker='o', linewidth=2.5, color='#e74c3c', markersize=6)
    ax.fill_between(promedio_hora.index, promedio_hora.values, alpha=0.2, color='#e74c3c')
    ax.axhline(y=15, color='green', linestyle='--', linewidth=1.5, label='OMS Límite Saludable (15 µg/m³)')
    ax.axhline(y=35, color='orange', linestyle='--', linewidth=1.5, label='Límite Moderado (35 µg/m³)')
    ax.set_title('🕐 Patrón Horario de PM2.5\n(Promedio por Hora del Día)', fontsize=13)
    ax.set_xlabel('Hora del Día')
    ax.set_ylabel('PM2.5 (µg/m³)')
    ax.set_xticks(range(0, 24))
    ax.legend()
    plt.tight_layout()
    plt.savefig('../docs/grafico_patron_horario.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('💾 Gráfico guardado en docs/')
else:
    print('⚠️ No hay datos de PM2.5 disponibles')

In [ ]:
# ── Gráfico 3: Heatmap por sensor y hora ──
df_pm25 = df[df['parametro'] == 'pm25']

if len(df_pm25) > 0:
    pivot = df_pm25.pivot_table(
        index='ubicacion',
        columns='hora',
        values='valor',
        aggfunc='mean'
    )
    
    fig, ax = plt.subplots(figsize=(14, 6))
    sns.heatmap(pivot, cmap='RdYlGn_r', annot=False, fmt='.1f',
                linewidths=0.3, ax=ax, cbar_kws={'label': 'PM2.5 (µg/m³)'})
    ax.set_title('🗺️ Heatmap: PM2.5 Promedio por Sensor y Hora', fontsize=13)
    ax.set_xlabel('Hora del Día')
    ax.set_ylabel('Sensor')
    plt.tight_layout()
    plt.savefig('../docs/grafico_heatmap.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('💾 Gráfico guardado en docs/')
else:
    print('⚠️ No hay datos suficientes para el heatmap')

In [ ]:
# ── Gráfico 4: Comparación de promedios por contaminante ──
promedios = df.groupby('parametro')['valor'].agg(['mean', 'std']).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
colores = sns.color_palette('husl', len(promedios))
bars = ax.bar(promedios['parametro'], promedios['mean'], 
              yerr=promedios['std'], capsize=5,
              color=colores, edgecolor='white', linewidth=1.2)
ax.set_title('📊 Concentración Promedio por Contaminante\n(con desviación estándar)', fontsize=13)
ax.set_xlabel('Contaminante')
ax.set_ylabel('Concentración Promedio')

for bar, val in zip(bars, promedios['mean']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../docs/grafico_comparacion.png', dpi=120, bbox_inches='tight')
plt.show()
print('💾 Gráfico guardado en docs/')

---
## 6️⃣ Conclusiones y Necesidad de Big Data

### ¿Qué datos genera el sistema?

Una red de monitoreo de calidad del aire genera:
- **Tipo:** Series de tiempo estructuradas + metadatos geoespaciales
- **Frecuencia:** Lectura cada 1–5 minutos por sensor
- **Volumen:** ~150 millones de registros/mes con 500 sensores

### ¿Quiénes usan estos datos?
- **Gobiernos:** alertas de emergencia y política ambiental
- **Hospitales:** correlacionar contaminación con ingresos respiratorios
- **Ciudadanos:** apps de calidad del aire en tiempo real
- **Investigadores:** modelos predictivos de salud pública

### ¿Por qué no alcanza Excel?

| Criterio | Excel/SQL | Big Data |
|---|---|---|
| **Volumen** | Colapsa con +1M filas | Escala a billones |
| **Velocidad** | Análisis por lotes | Tiempo real (streaming) |
| **Variedad** | Solo tablas planas | JSON, geodatos, series de tiempo |
| **Complejidad** | Fórmulas simples | ML, clustering, predicciones |

### 🏗️ Arquitectura Big Data ideal para este caso:
```
Sensores IoT → Apache Kafka (streaming) → Apache Spark (procesamiento)
     ↓
Data Lake (S3/HDFS) → ClickHouse/BigQuery → Dashboards / Alertas
```

In [ ]:
# Resumen final del análisis
print('=' * 50)
print('   📋 RESUMEN FINAL DEL ANÁLISIS')
print('=' * 50)
print(f'Total de registros analizados : {len(df):,}')
print(f'Parámetros monitoreados       : {df["parametro"].nunique()}')
print(f'Sensores activos              : {df["ubicacion"].nunique()}')
print(f'Ciudades cubiertas            : {df["ciudad"].nunique()}')

if 'fecha_utc' in df.columns:
    rango = df['fecha_utc'].max() - df['fecha_utc'].min()
    print(f'Rango temporal               : {rango.days} días')

print('=' * 50)
print('✅ Análisis completado')